In [1]:
%matplotlib inline
# Cell 1 — parameters

LAT, LON    = 16.8167, -2.9833   # canonical engine fixture (WO4b; matches all frozen TSVs)
RADIUS_KM   = 100
FROM_YEAR   = 1100
TO_YEAR     = 1200
FLOAT_TOL   = 1e-4               # tolerance for invariant assertions
L8_MIN      = 40                 # expected n_units lower bound for 100 km buffer at L8
L8_MAX      = 80                 # expected n_units upper bound

In [4]:
# Cell 2 — imports and output path

import sys
import warnings
warnings.filterwarnings('ignore', message='pandas only supports SQLAlchemy', category=UserWarning)

from pathlib import Path
import numpy as np
import pandas as pd
from scipy import stats

sys.path.insert(0, str(Path('.').resolve()))
import scripts.shared.db_utils as _dbu
from scripts.edop.areas.engine import areal_signature
from scripts.shared.db_utils import db_connect

ROOT = Path(_dbu.__file__).resolve().parents[2]
OUT  = ROOT / 'output' / 'edop' / 'areas'

In [5]:
# Cell 3 — run L6 and L8 payloads

conn = db_connect()
try:
    payload_l6 = areal_signature(LAT, LON, RADIUS_KM, conn, level=6,
                                  from_year=FROM_YEAR, to_year=TO_YEAR, include_detail=True)
    payload_l8 = areal_signature(LAT, LON, RADIUS_KM, conn, level=8,
                                  from_year=FROM_YEAR, to_year=TO_YEAR, include_detail=True)
finally:
    conn.close()

for label, p in [('L6', payload_l6), ('L8', payload_l8)]:
    nb     = p['neighborhood']
    basin  = [r for r in p['rows'] if r['band'] != 'T']
    t_rows = [r for r in p['rows'] if r['band'] == 'T']
    print(f'{label}: n_units={nb["n_units"]:3d}  basin_rows={len(basin):3d}  '
          f'T_rows={len(t_rows):3d}  shortfall={p["shortfall"]}')

print('done')

L6: n_units=  9  basin_rows= 51  T_rows=321  shortfall=0.0
L8: n_units= 74  basin_rows= 51  T_rows=321  shortfall=0.002459
done


In [6]:
# Cell 4 — sanity and shortfall check

n_l8 = payload_l8['neighborhood']['n_units']
assert L8_MIN <= n_l8 <= L8_MAX, f'L8 n_units={n_l8} outside expected range [{L8_MIN}, {L8_MAX}]'
print(f'OK  L8 n_units={n_l8} in expected range [{L8_MIN}, {L8_MAX}]')

sf6 = payload_l6['shortfall']
sf8 = payload_l8['shortfall']
# L8 shortfall is not expected to be identical to L6: finer polygons leave tiny geometry-
# precision slivers not present at L6. Flag only if shortfall is meaningfully large (>1%).
assert sf8 < 0.01, f'FAIL L8 shortfall={sf8} unexpectedly large (>1%)'
print(f'shortfall  L6={sf6}  L8={sf8:.6f}')
if abs(sf6 - sf8) > FLOAT_TOL:
    print(f'  NOTE: L8 shortfall non-zero ({sf8:.4%}) — geometry-precision slivers between '
          f'sub-basin polygons, not geographic absence. Not a bug.')

OK  L8 n_units=74 in expected range [40, 80]
shortfall  L6=0.0  L8=0.002459
  NOTE: L8 shortfall non-zero (0.2459%) — geometry-precision slivers between sub-basin polygons, not geographic absence. Not a bug.


In [8]:
# Cell 5 — Band T invariant: rows must be identical regardless of basin level
# The grid path uses the buffer geometry directly; level must not leak into it.

def _t_df(payload):
    return pd.DataFrame([
        {'variable': r['variable'], 'year': r['year'], 'epoch_year': r['epoch_year'],
         'score': r['representative_score'], 'n_units': r['n_units'],
         'coverage': r['coverage']}
        for r in payload['rows'] if r['band'] == 'T'
    ]).sort_values(['variable', 'year', 'epoch_year']).reset_index(drop=True)

t6 = _t_df(payload_l6)
t8 = _t_df(payload_l8)

assert len(t6) == len(t8), f'FAIL Band T row count: L6={len(t6)}  L8={len(t8)}'
s6 = pd.to_numeric(t6['score'], errors='coerce').fillna(0)
s8 = pd.to_numeric(t8['score'], errors='coerce').fillna(0)
score_diff  = (s6 - s8).abs().max()
nunits_diff = (t6['n_units'] - t8['n_units']).abs().max()
assert score_diff  < FLOAT_TOL, f'FAIL Band T score mismatch: max_diff={score_diff}'
assert nunits_diff == 0,        f'FAIL Band T n_units mismatch: max_diff={nunits_diff}'

print(f'OK  Band T invariant: {len(t6)} rows identical L6 vs L8')
print(f'    max score delta={score_diff:.2e}  max n_units delta={nunits_diff}')

OK  Band T invariant: 321 rows identical L6 vs L8
    max score delta=0.00e+00  max n_units delta=0


In [13]:
# Cell 6 — build comparison DataFrame (basin rows only)

def flatten_basin(payload, label):
    rows = []
    for r in payload['rows']:
        if r['band'] == 'T':
            continue
        d = r.get('detail') or {}
        rows.append({
            'variable':              r['variable'],
            'band':                  r['band'],
            'method':                r['method'],
            f'score_{label}':        r['representative_score'],
            f'n_units_{label}':      r['n_units'],
            f'coherence_{label}':    r['coherence'],
            f'modality_{label}':     r.get('modality'),
            f'status_{label}':       r['status'],
            f'wtz_{label}':          d.get('weight_at_zero'),
        })
    return pd.DataFrame(rows)

df6 = flatten_basin(payload_l6, 'l6')
df8 = flatten_basin(payload_l8, 'l8')

val_cols_l8 = [c for c in df8.columns if c.endswith('_l8')]
cmp = df6.merge(df8[['variable'] + val_cols_l8], on='variable', how='outer')
cmp['score_delta'] = (pd.to_numeric(cmp['score_l8'], errors='coerce')
                    - pd.to_numeric(cmp['score_l6'], errors='coerce')).round(3)

def _meaningful_flip(a, b):
    """True only when both values are non-null AND they differ."""
    return a.notna() & b.notna() & (a != b)

cmp['coherence_flip'] = _meaningful_flip(cmp['coherence_l6'], cmp['coherence_l8'])
cmp['modality_flip']  = _meaningful_flip(cmp['modality_l6'],  cmp['modality_l8'])
cmp['status_flip']    = _meaningful_flip(cmp['status_l6'],    cmp['status_l8'])

print(f'Basin rows: L6={len(df6)}  L8={len(df8)}  merged={len(cmp)}')
print(f'coherence flips (meaningful): {cmp["coherence_flip"].sum()}')
print(f'modality flips  (meaningful): {cmp["modality_flip"].sum()}')
print(f'status flips    (meaningful): {cmp["status_flip"].sum()}')

Basin rows: L6=51  L8=51  merged=51
coherence flips (meaningful): 4
modality flips  (meaningful): 11
status flips    (meaningful): 0


In [10]:
# Cell 7 — B1 score deltas and rank-order stability (Spearman ρ)

b1 = cmp[cmp['method'] == 'area_weighted'].copy()
b1_scored = b1.dropna(subset=['score_l6', 'score_l8'])

rho, pval = stats.spearmanr(b1_scored['score_l6'], b1_scored['score_l8'])
print(f'B1 area_weighted: {len(b1_scored)} scorable vars of {len(b1)} total')
print(f'Spearman ρ = {rho:.4f}  (p = {pval:.2e})')
print()

cols = ['variable', 'score_l6', 'score_l8', 'score_delta',
        'coherence_l6', 'coherence_l8', 'coherence_flip']
print(b1_scored[cols].sort_values('score_delta', key=abs, ascending=False).to_string(index=False))

B1 area_weighted: 6 scorable vars of 34 total
Spearman ρ = 1.0000  (p = 0.00e+00)

       variable  score_l6  score_l8  score_delta coherence_l6 coherence_l8  coherence_flip
       elev_min     63.20     51.95       -11.25 concentrated concentrated           False
       elev_max     26.13     28.27         2.14 concentrated concentrated           False
stream_gradient      5.57      6.44         0.87 concentrated concentrated           False
        aridity     10.19     10.60         0.41 concentrated concentrated           False
        temp_yr     97.85     98.01         0.16 concentrated concentrated           False
      precip_yr     16.11     16.11         0.00 concentrated concentrated           False


In [14]:
# Cell 8 — flip inventory: coherence, modality, status

print('=== COHERENCE FLIPS (meaningful only) ===')
flips_c = cmp[cmp['coherence_flip']][['variable', 'method', 'coherence_l6', 'coherence_l8']]
print(flips_c.to_string(index=False) if len(flips_c) else '  none')

print()
print('=== MODALITY FLIPS (meaningful only) ===')
flips_m = cmp[cmp['modality_flip']][['variable', 'method', 'modality_l6', 'modality_l8']]
print(flips_m.to_string(index=False) if len(flips_m) else '  none')
print('  (watch: temp_yr_upstream and pct_sand — flagged known-weak in register)')

print()
print('=== STATUS FLIPS ===')
flips_s = cmp[cmp['status_flip']][['variable', 'method', 'status_l6', 'status_l8',
                                    'score_l6', 'score_l8']]
print(flips_s.to_string(index=False) if len(flips_s) else '  none')

print()
print('=== dist_sink: catalog zero_fraction + detail inspection ===')
# Check catalog zero_fraction for dist_sink
from scripts.edop.areas.engine import load_catalog
cat = load_catalog(level=6)
if 'dist_sink' in cat.index:
    row = cat.loc['dist_sink']
    print(f'  catalog: zero_fraction={row["zero_fraction"]}  kind={row["kind"]}  '
          f'position_method={row["position_method"]}')
else:
    print(f'  dist_sink not in catalog index; keys containing sink: '
          f'{[k for k in cat.index if "sink" in k.lower()]}')

# Print full detail dict for dist_sink at both levels
for label, p in [('L6', payload_l6), ('L8', payload_l8)]:
    row = next((r for r in p['rows'] if r['variable'] == 'dist_sink'), None)
    if row:
        print(f'  {label}: score={row["representative_score"]}  coherence={row["coherence"]}  '
              f'detail={row.get("detail")}')

=== COHERENCE FLIPS (meaningful only) ===
          variable        method coherence_l6 coherence_l8
  aridity_upstream area_weighted       spread concentrated
   cropland_extent area_weighted concentrated       spread
precip_yr_upstream area_weighted       spread concentrated
         slope_avg area_weighted concentrated       spread

=== MODALITY FLIPS (meaningful only) ===
                variable        method modality_l6 modality_l8
        aridity_upstream area_weighted  two_regime    unimodal
         cropland_extent area_weighted  two_regime    unimodal
cropland_extent_upstream area_weighted  two_regime    unimodal
      human_footprint_09 area_weighted  two_regime    unimodal
 pasture_extent_upstream area_weighted  two_regime    unimodal
                pct_sand area_weighted  two_regime    unimodal
      precip_yr_upstream area_weighted  two_regime    unimodal
        temp_yr_upstream area_weighted  two_regime    unimodal
            wet_pct_grp1 area_weighted  two_regime    

In [15]:
# Cell 9 — B2 dominant basin stability
# The dominant hybas_id will differ (different polygons at L8).
# The question is whether discharge value and perennial verdict are stable.

print('B2 dominant basin:')
for label, p in [('L6', payload_l6), ('L8', payload_l8)]:
    b2_rows = [r for r in p['rows'] if r['method'] == 'dominant_basin']
    for r in b2_rows:
        d = r.get('detail') or {}
        print(f'  {label}  {r["variable"]:<20}  '
              f'dominant_id={d.get("dominant_hybas_id")}  '
              f'score={r["representative_score"]}  '
              f'raw={r["representative_raw"]}  '
              f'perennial={d.get("perennial")}')

B2 dominant basin:
  L6  discharge_yr          dominant_id=1060564960  score=86.01  raw=567.595  perennial=None
  L6  discharge_max         dominant_id=1060564960  score=83.73  raw=1089.243  perennial=None
  L6  discharge_min         dominant_id=1060564960  score=89.91  raw=301.824  perennial=True
  L8  discharge_yr          dominant_id=1080572190  score=95.24  raw=601.4  perennial=None
  L8  discharge_max         dominant_id=1080572190  score=94.36  raw=1085.528  perennial=None
  L8  discharge_min         dominant_id=1080572190  score=96.93  raw=361.908  perennial=True


In [16]:
# Cell 10 — B5 river_area extreme carrier
# At L6: carrier (Inner Niger Delta) ≠ B2 discharge dominant.
# Does that split persist at L8?

for label, p in [('L6', payload_l6), ('L8', payload_l8)]:
    ext = next((r for r in p['rows'] if r['method'] == 'extreme'), None)
    b2  = next((r for r in p['rows'] if r['method'] == 'dominant_basin'
                and r['variable'] == 'discharge_yr'), None)
    if ext and b2:
        carrier = (ext.get('detail') or {}).get('dominant_hybas_id')
        b2_dom  = (b2.get('detail') or {}).get('dominant_hybas_id')
        split   = carrier != b2_dom
        print(f'{label}: river_area carrier={carrier}  '
              f'discharge_yr dominant={b2_dom}  '
              f'split={split}  '
              f'(raw={ext["representative_raw"]} km²)')

L6: river_area carrier=1060582960  discharge_yr dominant=1060564960  split=True  (raw=4273.403 km²)
L8: river_area carrier=1080551560  discharge_yr dominant=1080572190  split=True  (raw=1257.882 km²)


In [17]:
# Cell 11 — B4/B1 cross-block consistency
# endorheic basin fraction (B4 outlet_type mixture) ≈ dist_sink low-regime weight (B1/B6)
# dist_sink is two_regime at both levels; weight_at_zero lives in detail['regimes'][0]['weight'].
# Exact agreement expected only when shortfall ≈ 0 (holds here).
# NOTE: at L8 the low-regime center shifts from 0.0 → 12.99, meaning the detector groups
# some low-but-non-zero basins with the endorheic zeros — agreement will be approximate.

print('Cross-block consistency: endorheic share (B4 outlet_type) vs dist_sink low-regime weight (B1/B6)')
print()
for label, p in [('L6', payload_l6), ('L8', payload_l8)]:
    # B4: outlet_type mixture — find the endorheic class
    ot = next((r for r in p['rows'] if r['variable'] == 'outlet_type'), None)
    mixture = (ot.get('detail') or {}).get('mixture', []) if ot else []
    print(f'  {label} outlet_type mixture:')
    for m in mixture:
        print(f'    class_id={m["class_id"]}  weight={m["weight"]:.4f}  label={m.get("label", "?")}')

    # B1/B6: dist_sink low-regime weight
    ds = next((r for r in p['rows'] if r['variable'] == 'dist_sink'), None)
    regimes = (ds.get('detail') or {}).get('regimes', []) if ds else []
    if regimes:
        r0 = regimes[0]
        print(f'  {label} dist_sink regime 0: center={r0["center"]:.2f}  weight={r0["weight"]:.4f}')
    else:
        print(f'  {label} dist_sink: no regimes in detail')
    print()

Cross-block consistency: endorheic share (B4 outlet_type) vs dist_sink low-regime weight (B1/B6)

  L6 outlet_type mixture:
    class_id=0  weight=0.5346  label=?
    class_id=20  weight=0.4654  label=?
  L6 dist_sink regime 0: center=0.00  weight=0.4654

  L8 outlet_type mixture:
    class_id=0  weight=0.5316  label=?
    class_id=10  weight=0.3230  label=?
    class_id=20  weight=0.1429  label=?
  L8 dist_sink regime 0: center=12.99  weight=0.4671



In [18]:
# Cell 12 — save comparison TSV

out_cols = ['variable', 'band', 'method',
            'score_l6', 'score_l8', 'score_delta',
            'n_units_l6', 'n_units_l8',
            'coherence_l6', 'coherence_l8', 'coherence_flip',
            'modality_l6',  'modality_l8',  'modality_flip',
            'status_l6',    'status_l8',    'status_flip']

out_path = OUT / 'wo12_l6_l8_comparison.tsv'
cmp[out_cols].to_csv(out_path, sep='\t', index=False)
print(f'Saved {len(cmp)} rows → {out_path}')
print()
print('Summary:')
print(cmp.groupby('method')[['coherence_flip', 'modality_flip', 'status_flip']].sum().to_string())

Saved 51 rows → /Users/karlg/Documents/repos/_edops/output/edop/areas/wo12_l6_l8_comparison.tsv

Summary:
                   coherence_flip  modality_flip  status_flip
method                                                       
area_weighted                   4             11            0
class_mixture                   0              0            0
distribution_only               0              0            0
dominant_basin                  0              0            0
extreme                         0              0            0
flag_fraction                   0              0            0


In [19]:
# Cell 13 — seam-alignment check for the 11 modality flippers
#
# A variable is seam-aligned if its two L6 regimes track the endorheic/exorheic split:
# one regime weight ≈ endorheic fraction (0.4654), the other ≈ exorheic fraction (0.5346).
# ENDO_WT is the L6 endorheic fraction from dist_sink regime 0 (cell 8 detail).
# SEAM_TOL: how close a regime weight must be to ENDO_WT to count as seam-aligned.
# A value of 0.10 (10 pp) is permissive; tighten to 0.05 for strict reading.

ENDO_WT  = 0.4654   # endorheic fraction at L6 (dist_sink regime 0 weight)
SEAM_TOL = 0.10     # ±10 pp tolerance

# The 11 variables that flipped two_regime → unimodal
flipped = set(cmp[cmp['modality_flip']]['variable'])

# Load L6 regimes TSV
regimes_df = pd.read_csv(OUT / 'step3_block6_regimes.tsv', sep='\t')

print(f'Seam-alignment of the 11 modality flippers (ENDO_WT={ENDO_WT}, TOL=±{SEAM_TOL})')
print(f'{"variable":<30} {"w0":>6} {"w1":>6}  {"min|w-endo|":>12}  seam_aligned')
print('-' * 72)

results = []
for var, grp in regimes_df[regimes_df['variable'].isin(flipped)].groupby('variable'):
    weights = sorted(grp['regime_weight'].tolist())
    if len(weights) < 2:
        continue
    w0, w1 = weights[0], weights[1]
    closest = min(abs(w0 - ENDO_WT), abs(w1 - ENDO_WT))
    aligned = closest <= SEAM_TOL
    results.append({'variable': var, 'w0': w0, 'w1': w1, 'closest': closest, 'aligned': aligned})
    print(f'{var:<30} {w0:>6.4f} {w1:>6.4f}  {closest:>12.4f}  {"YES" if aligned else "no"}')

n_aligned = sum(r['aligned'] for r in results)
print()
print(f'Seam-aligned: {n_aligned} of {len(results)}  |  Not aligned: {len(results) - n_aligned} of {len(results)}')
print()

# dist_sink (did NOT flip — stayed two_regime at both levels) as reference
ds_grp = regimes_df[regimes_df['variable'] == 'dist_sink']
if len(ds_grp):
    dw = sorted(ds_grp['regime_weight'].tolist())
    print(f'Reference — dist_sink (survived, genuinely bimodal): w0={dw[0]:.4f}  w1={dw[1]:.4f}  '
          f'closest={min(abs(dw[0]-ENDO_WT), abs(dw[1]-ENDO_WT)):.4f}')

Seam-alignment of the 11 modality flippers (ENDO_WT=0.4654, TOL=±0.1)
variable                           w0     w1   min|w-endo|  seam_aligned
------------------------------------------------------------------------
aridity_upstream               0.2922 0.7078        0.1732  no
cropland_extent                0.2043 0.7957        0.2611  no
cropland_extent_upstream       0.3103 0.6897        0.1551  no
human_footprint_09             0.4654 0.5346        0.0000  YES
pasture_extent_upstream        0.4654 0.5346        0.0000  YES
pct_sand                       0.4605 0.5395        0.0049  YES
precip_yr_upstream             0.2922 0.7078        0.1732  no
temp_yr_upstream               0.2922 0.7078        0.1732  no
wet_pct_grp1                   0.4470 0.5530        0.0184  YES
wet_pct_grp1_upstream          0.4470 0.5530        0.0184  YES
wet_pct_grp2_upstream          0.4470 0.5530        0.0184  YES

Seam-aligned: 6 of 11  |  Not aligned: 5 of 11

Reference — dist_sink (survived, gen

In [20]:
# Cell 14 — WO13 Part 1: does an absolute separation floor exist?
#
# For all 12 two_regime variables at L6 (11 flippers + dist_sink), compute the
# regime-center gap in percentile points.  Report sorted; answer yes/no on
# whether one floor value A separates dist_sink from all 11 flippers.
# regimes_df and flipped are carried from cell 13.

gaps = (
    regimes_df.groupby('variable')['regime_center']
    .agg(lambda x: x.max() - x.min())
    .rename('gap_pp')
    .reset_index()
    .sort_values('gap_pp')
    .reset_index(drop=True)
)

gaps['role'] = gaps['variable'].apply(
    lambda v: 'SURVIVOR' if v == 'dist_sink' else 'flipper'
)

print(f'Regime-center gaps for all {len(gaps)} two_regime variables at L6 (percentile points)')
print(f'{"variable":<32} {"gap_pp":>8}  role')
print('-' * 54)
for _, row in gaps.iterrows():
    marker = '  <-- dist_sink' if row['variable'] == 'dist_sink' else ''
    print(f'{row["variable"]:<32} {row["gap_pp"]:>8.2f}  {row["role"]}{marker}')

# Is there a clean floor?
flipper_gaps = gaps[gaps['role'] == 'flipper']['gap_pp']
survivor_gap = gaps[gaps['role'] == 'SURVIVOR']['gap_pp'].values[0]
flipper_max  = flipper_gaps.max()
window       = survivor_gap - flipper_max

print()
if window > 0:
    print(f'YES — clean separation exists.')
    print(f'  Largest flipper gap : {flipper_max:.2f} pp')
    print(f'  dist_sink gap       : {survivor_gap:.2f} pp')
    print(f'  Separating window   : {window:.2f} pp')
    print(f'  Any A in ({flipper_max:.1f}, {survivor_gap:.1f}) pp separates the 12 cases.')
else:
    print(f'NO — overlap: largest flipper ({flipper_max:.2f} pp) >= dist_sink ({survivor_gap:.2f} pp)')
    print('  A single value floor cannot separate the cases.')

Regime-center gaps for all 12 two_regime variables at L6 (percentile points)
variable                           gap_pp  role
------------------------------------------------------
temp_yr_upstream                     7.10  flipper
pct_sand                            17.89  flipper
cropland_extent                     25.91  flipper
aridity_upstream                    29.03  flipper
human_footprint_09                  30.79  flipper
cropland_extent_upstream            42.87  flipper
precip_yr_upstream                  58.49  flipper
wet_pct_grp2_upstream               63.59  flipper
pasture_extent_upstream             65.93  flipper
wet_pct_grp1_upstream               68.87  flipper
wet_pct_grp1                        80.54  flipper
dist_sink                           82.51  SURVIVOR  <-- dist_sink

YES — clean separation exists.
  Largest flipper gap : 80.54 pp
  dist_sink gap       : 82.51 pp
  Separating window   : 1.97 pp
  Any A in (80.5, 82.5) pp separates the 12 cases.
